# Modelado de datos FINANOM

Objetivo: convertir la base limpia en un dataset listo para detección no supervisada de anomalías financieras con Isolation Forest.

La unidad de análisis se mantiene igual que en las fases anteriores: **una fila = una transacción**. No se eliminan filas porque los casos raros son precisamente candidatos a anomalía.

Entregables de esta fase:

- `data_modeling/data_modeling.py`: pipeline reproducible.
- `data_modeling/output/transacciones_modelado.parquet`: dataset final con trazabilidad + features.
- `data_modeling/output/X_modelo.parquet`: matriz numérica pura para entrenamiento.
- `data_modeling/diccionario_modelado.md`: diccionario final.
- `data_modeling/output/reporte_calidad_modelado.md`: evaluación recursiva de calidad.

## Setup

In [1]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "data_modeling":
    ROOT = ROOT.parent

clean_path = ROOT / "data_cleaning" / "output" / "transacciones_limpio.parquet"
modeled_path = ROOT / "data_modeling" / "output" / "transacciones_modelado.parquet"
x_path = ROOT / "data_modeling" / "output" / "X_modelo.parquet"
diagnostics_path = ROOT / "data_modeling" / "output" / "diagnosticos_modelado.json"
importance_path = ROOT / "data_modeling" / "output" / "proxy_feature_importance.csv"
anomaly_sample_path = ROOT / "data_modeling" / "output" / "proxy_anomaly_sample.csv"

## Punto de partida

La fase de limpieza dejó 34 columnas: identificadores de trazabilidad, categóricas, numéricas financieras, flags, fechas y texto libre. En esta fase esas columnas se transforman en dos familias:

- `trace_*`: columnas para ubicar la transacción original y poder explicarla al auditor. **No entran al modelo**.
- `feat_*`: columnas numéricas, sin nulos, listas para Isolation Forest.

In [2]:
clean = pd.read_parquet(clean_path)
print(clean.shape)
clean.dtypes.to_frame("tipo").join(
    (100 * clean.isna().mean()).round(2).to_frame("pct_nulos")
).head(40)

(1145526, 34)


,tipo,pct_nulos
t_folio,string,0.00
t_folio_ext,string,0.00
t_referencia,string,0.00
t_transaccion,string,0.00
t_cve_res,string,21.44
t_cuarto,string,0.00
t_codigo,string,0.00
t_carabo,string,0.00
t_usuario,string,0.00
t_usuario_mod,string,80.37


## Fase 1 — Modelado de datos base

Transformaciones aplicadas por `data_modeling.py`:

1. **Trazabilidad separada**: folio, referencia, transacción, reservación, cuarto, código y timestamp se conservan como `trace_*`, pero no se usan como features crudas.
2. **Flags operativos y financieros**: abono, cancelación, split, renta, reservación enlazada, modificación de usuario y reglas de signo.
3. **Montos transformados**: `log1p(abs(x))` para colas largas y robust z por (`t_codigo`, `t_carabo`) para comparar cada concepto en su propio contexto.
4. **Ratios financieros**: impuesto/monto, propina/monto y monto contra tarifa diaria/total.
5. **Contexto de reservación**: ocupación, noches, tarifa y depósito se imputan a mediana y se escalan con mediana/IQR.
6. **Fechas**: hora, día de semana y mes se codifican con seno/coseno; también se crean días relativos a llegada/salida.
7. **Duplicados**: conteos por folio/subfolio/código/monto/día y por minuto para atacar directamente el dolor del negocio.
8. **Categóricas**: encoding por frecuencia relativa, sin bucket de “otros”, para que categorías raras sigan visibles.
9. **Texto libre**: no se exporta crudo por posible PII; se reemplaza por longitud y keywords operativas.

In [3]:
modeled = pd.read_parquet(modeled_path)
feature_cols = [c for c in modeled.columns if c.startswith("feat_")]
trace_cols = [c for c in modeled.columns if c.startswith("trace_")]

summary = {
    "filas": len(modeled),
    "columnas_totales": modeled.shape[1],
    "trace_cols": len(trace_cols),
    "feature_cols": len(feature_cols),
    "nulos_features": int(modeled[feature_cols].isna().sum().sum()),
    "features_no_numericas": [c for c in feature_cols if not pd.api.types.is_numeric_dtype(modeled[c])],
    "features_constantes": [c for c in feature_cols if modeled[c].nunique(dropna=False) <= 1],
}
summary

{'filas': 1145526,
 'columnas_totales': 72,
 'trace_cols': 9,
 'feature_cols': 63,
 'nulos_features': 0,
 'features_no_numericas': [],
 'features_constantes': []}

In [4]:
modeled[trace_cols + feature_cols[:8]].head()

,trace_row_id,trace_t_folio,trace_t_folio_ext,trace_t_referencia,trace_t_transaccion,trace_t_cve_res,trace_t_cuarto,trace_t_codigo,trace_t_timestamp,feat_es_abono,feat_cargo_cancelado,feat_cancelacion_sin_marca,feat_es_split,feat_es_renta,feat_tiene_reservacion,feat_usuario_modificado,feat_usuario_mod_distinto
0,0,7518,0,A1320A,213,I 41791 1,13207,PROPTI,2021-06-28 05:31:00,0,1,0,0,1,1,1,1
1,1,7338,0,1320H2,214,G 199 17,13208,RENHAB,2021-06-28 05:31:00,0,0,0,0,1,1,0,0
2,2,7338,0,1320A2,215,G 199 17,13208,PROPTI,2021-06-28 05:31:00,0,0,0,0,1,1,0,0
3,3,7548,0,1320H2,216,I 41177 1,13209,RENHAB,2021-06-28 05:31:00,0,0,0,0,1,1,0,0
4,4,7548,0,1320A2,217,I 41177 1,13209,PROPTI,2021-06-28 05:31:00,0,0,0,0,1,1,0,0


## Fase 2 — Evaluación recursiva de calidad

Se aplicaron tres diagnósticos:

- **Nulos/tipos/constantes**: el dataset final debe tener features numéricas, sin nulos y con varianza.
- **Correlación alta**: se eliminan redundancias con `|corr| >= 0.985` cuando una feature duplica a otra más interpretable.
- **Isolation Forest proxy**: se entrena un IF rápido sobre muestra de 120k filas para revisar qué features son usadas en los splits. No es el modelo final; es diagnóstico de señal.

In [5]:
diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
{
    "features_candidatas": diagnostics["candidate_features"],
    "features_finales": diagnostics["final_features"],
    "features_eliminadas": diagnostics["dropped_features"],
    "pares_alta_corr_finales": diagnostics["final_high_correlation_pairs"],
    "score_proxy": diagnostics["anomaly_score_summary"],
}

{'features_candidatas': 70,
 'features_finales': 63,
 'features_eliminadas': {'feat_falta_contexto_reserva': 'Inverso perfecto de `feat_tiene_reservacion`; se conserva la version positiva.',
  'feat_t_usuario_mod_missing': 'Inverso perfecto de `feat_usuario_modificado`; se conserva la version positiva.',
  'feat_impuesto_signo_distinto_monto': 'Constante en la base actual; no aporta separacion al Isolation Forest.',
  'feat_propina_mayor_monto': 'Constante en la base actual; no aporta separacion al Isolation Forest.',
  'feat_t_usuario_mod_freq': 'Correlacion >= 0.985 con `feat_usuario_modificado`; la frecuencia queda dominada por el nulo.',
  'feat_desvio_iva_16_abs': 'Correlacion >= 0.985 con `feat_impuesto_ratio_abs`; se conserva el ratio fiscal mas general.',
  'feat_cargo_despues_salida': 'Correlacion >= 0.985 con `feat_cargo_fuera_estancia`; se conserva la regla agregada.'},
 'pares_alta_corr_finales': [],
 'score_proxy': {'min': -0.1609856867775527,
  'p50': -0.12326119122522974

### Visualización — Importancia proxy

La importancia se calcula por frecuencia de uso de cada feature en los splits del Isolation Forest proxy. Es una señal práctica de qué variables ayudan a aislar transacciones raras, no una explicación causal.

![Top 20 proxy features](output/proxy_feature_importance_top20.png)

In [6]:
importance = pd.read_csv(importance_path)
importance.head(20)

,feature,split_count,split_importance
0,feat_hora_sin,3270,0.036415
1,feat_hora_cos,3159,0.035179
2,feat_monto_abs_log,2952,0.032874
3,feat_monto_z_codigo_carabo,2950,0.032852
4,feat_folio_total_movimientos_log,2867,0.031927
5,feat_mes_sin,2788,0.031047
6,feat_mes_cos,2741,0.030524
7,feat_folio_dia_movimientos_log,2737,0.030480
8,feat_t_usuario_freq,2703,0.030101
9,feat_dia_semana_sin,2589,0.028831


### Visualización — Distribución de score proxy

El percentil 98 corresponde a la contaminación proxy del 2%. Para el demo, este umbral es útil como punto inicial, pero después debe calibrarse con revisión del auditor.

![Distribución proxy scores](output/proxy_anomaly_score_distribution.png)

In [7]:
pd.read_csv(anomaly_sample_path).head(20)

,trace_row_id,trace_t_folio,trace_t_codigo,trace_t_timestamp,proxy_anomaly_score
0,228797,17256,TRANSF,2021-12-08 12:59:00,0.132677
1,26481,352,CANCXC,2021-03-12 15:43:00,0.124235
2,562014,48708,XFAC,2023-02-18 11:57:00,0.122248
3,22492,30,XFAC,2021-03-20 07:06:00,0.119914
4,973617,106449,RENAJU,2024-10-27 07:12:00,0.110060
5,193060,6343,RENHAB,2021-10-28 01:58:00,0.107031
6,261108,142232,RENCOM,2025-12-15 12:48:00,0.105991
7,781021,78895,CANXFA,2023-12-26 11:31:00,0.102371
8,134076,6343,RENHAB,2021-07-26 05:22:00,0.101271
9,120401,6343,RENHAB,2021-07-12 04:17:00,0.100799


## Decisiones después de iterar

Ajustes hechos después del primer dataset candidato:

- Se eliminaron inversos perfectos: `feat_falta_contexto_reserva` y `feat_t_usuario_mod_missing`.
- Se eliminaron constantes en esta base: `feat_impuesto_signo_distinto_monto` y `feat_propina_mayor_monto`.
- Se eliminaron tres redundancias por correlación alta: `feat_t_usuario_mod_freq`, `feat_desvio_iva_16_abs` y `feat_cargo_despues_salida`.
- Se conservaron flags raros como `feat_es_split`, `feat_cargo_antes_llegada` y keywords de texto porque son pocos casos, pero tienen lectura directa para auditoría.
- No se usó VIF como criterio principal: las features mezclan ciclos, flags, frecuencias y transformaciones no lineales; para un modelo basado en árboles, el filtro de correlación alta es suficiente y más interpretable.

## Resultado final

El dataset queda listo para modelado:

- 1,145,526 transacciones.
- 63 features finales.
- 0 nulos en features.
- 0 features constantes.
- 0 pares finales con `|corr| >= 0.985`.

Para entrenar, usar directamente `X_modelo.parquet` o seleccionar columnas `feat_*` desde `transacciones_modelado.parquet`.

In [8]:
from sklearn.ensemble import IsolationForest

X = pd.read_parquet(x_path)
model = IsolationForest(
    n_estimators=200,
    max_samples=min(8192, len(X)),
    contamination=0.02,
    random_state=42,
    n_jobs=-1,
)
# model.fit(X)  # Descomentar en la fase de entrenamiento formal.